In [ ]:
import pandas as pd, numpy as np, json, torch, torch.nn as nn
from sklearn.linear_model import LogisticRegression

full_preds = pd.read_parquet("../processed/full_preds.parquet")
snapshot_features = pd.read_parquet("../processed/snapshot_features.parquet")
plays = pd.read_csv("../nfl-big-data-bowl-2023/plays.csv")

In [ ]:
lstm_preds = pd.read_parquet("../processed/lstm_predictions.parquet")
raw_proba = lstm_preds["blocker_lost_predicted_proba"].values.reshape(-1, 1)
true_label = lstm_preds["blocker_lost_actual"].values

calibrator = LogisticRegression()
calibrator.fit(raw_proba, true_label)

In [ ]:
selection_pool = full_preds.merge(
    snapshot_features[[
        "gameId", "playId", "nflId", "rusher_nflId",
        "dist_snap", "dist_final", "orientation_diff_snap", "orientation_diff_final",
        "blocker_position", "rusher_position"
    ]],
    on=["gameId", "playId", "nflId", "rusher_nflId"],
    how="left",
)

selection_pool["movement_score"] = (
    (selection_pool["orientation_diff_final"] - selection_pool["orientation_diff_snap"]).abs()
    + (selection_pool["dist_final"] - selection_pool["dist_snap"]).abs()
)

selection_pool = selection_pool.merge(
    plays[["gameId", "playId", "playDirection"]] if "playDirection" in plays.columns
    else plays[["gameId", "playId"]],
    on=["gameId", "playId"], how="left",
)

print(f"{len(selection_pool):,}")

In [ ]:
def pick_spread_by_position(df, n, position_col="blocker_position"):
    oline = {"C", "G", "T"}
    df = df[df[position_col].isin(oline)]

    picks = []
    positions = ["C", "G", "T"]
    per_position = n // len(positions)
    for pos in positions:
        picks.append(df[df[position_col] == pos].head(per_position))
    remaining = n - sum(len(p) for p in picks)
    if remaining > 0:
        already_picked_idx = pd.concat(picks).index
        picks.append(df[~df.index.isin(already_picked_idx)].head(remaining))
    return pd.concat(picks).drop_duplicates(subset=["gameId", "playId", "nflId", "rusher_nflId"])

In [ ]:
confident_wins = selection_pool[
    (selection_pool["blocker_lost_actual"] == 0) &
    (selection_pool["blocker_win_probability"] >= 0.95)
].sort_values("blocker_win_probability", ascending=False)

confident_losses = selection_pool[
    selection_pool["blocker_lost_actual"] == 1
].sort_values("blocker_win_probability", ascending=True)

high_movement = selection_pool.sort_values("movement_score", ascending=False)

wins_selected = pick_spread_by_position(confident_wins, 10)
losses_selected = pick_spread_by_position(confident_losses, 10)
swings_selected = pick_spread_by_position(high_movement, 10)

In [ ]:
already_selected_idx = pd.concat([wins_selected, losses_selected, swings_selected]).drop_duplicates(
    subset=["gameId", "playId", "nflId", "rusher_nflId"]
).index
remaining_pool = selection_pool[~selection_pool.index.isin(already_selected_idx)]

blocker_snap_counts = remaining_pool[remaining_pool["blocker_position"].isin({"C", "G", "T"})].groupby("nflId").size().sort_values(ascending=False)
top_snap_blockers = blocker_snap_counts.head(10).index.tolist()
wildcard_candidates = remaining_pool[remaining_pool["nflId"].isin(top_snap_blockers)]
wildcard_high_profile = wildcard_candidates.sample(3, random_state = 42)

notable_rushers = ["Aaron Donald", "Chris Jones", "Myles Garrett", "T.J. Watt", "Cameron Heyward"]
notable_matches = remaining_pool[remaining_pool["rusherName"].isin(notable_rushers)]
wildcard_notable = notable_matches.sample(n=7, random_state=42)

wildcards_selected = pd.concat([wildcard_high_profile, wildcard_notable]).drop_duplicates(
        subset=["gameId", "playId", "nflId", "rusher_nflId"]
)

In [ ]:
base_selection = pd.concat([
    wins_selected.assign(category="win"),
    losses_selected.assign(category="loss"),
    swings_selected.assign(category="movement_swing"),
    wildcards_selected.assign(category="wildcard"),
]).drop_duplicates(subset=["gameId", "playId", "nflId", "rusher_nflId"]).reset_index(drop=True)

oline = {"C", "G", "T"}
non_ol = base_selection[~base_selection["blocker_position"].isin(oline)]
print(len(non_ol)==0)
print(base_selection["category"].value_counts())

In [ ]:
already_base_idx = base_selection.set_index(["gameId","playId","nflId","rusher_nflId"]).index
prob_swing_candidates = (
    selection_pool[
        (~selection_pool.set_index(["gameId","playId","nflId","rusher_nflId"]).index.isin(already_base_idx)) &
        (selection_pool["blocker_position"].isin({"C","G","T"}))
    ]
    .sort_values("movement_score", ascending=False)
    .head(400)
)

In [ ]:
%run 01_ingest.ipynb

blockers = tracking_df[tracking_df["pff_role"] == "Pass Block"].copy()

rusher_cols = ["gameId", "playId", "frameId", "nflId", "x", "y", "s", "a", "o", "dir"]
rushers = tracking_df[rusher_cols].rename(columns={
    "nflId": "rusher_nflId",
    "x": "rusher_x", "y": "rusher_y",
    "s": "rusher_s", "a": "rusher_a",
    "o": "rusher_o", "dir": "rusher_dir",
})

paired_prob_swing = blockers.merge(
    rushers,
    left_on=["gameId", "playId", "frameId", "pff_nflIdBlockedPlayer"],
    right_on=["gameId", "playId", "frameId", "rusher_nflId"],
    how="inner",
)

prob_swing_keys = set(zip(
    prob_swing_candidates["gameId"], prob_swing_candidates["playId"],
    prob_swing_candidates["nflId"], prob_swing_candidates["rusher_nflId"]
))
paired_prob_swing["key"] = list(zip(paired_prob_swing["gameId"], paired_prob_swing["playId"], paired_prob_swing["nflId"], paired_prob_swing["rusher_nflId"]))
paired_prob_swing = paired_prob_swing[paired_prob_swing["key"].isin(prob_swing_keys)].drop(columns=["key"])

paired_prob_swing["distance"] = np.sqrt(
    (paired_prob_swing["x"] - paired_prob_swing["rusher_x"])**2 + (paired_prob_swing["y"] - paired_prob_swing["rusher_y"])**2
)
paired_prob_swing = paired_prob_swing.sort_values(["gameId", "playId", "nflId", "frameId"])
paired_prob_swing["prev_distance"] = paired_prob_swing.groupby(["gameId", "playId", "nflId"])["distance"].shift(1)
paired_prob_swing["closing_speed"] = (paired_prob_swing["prev_distance"] - paired_prob_swing["distance"]).fillna(0)
paired_prob_swing["orientation_diff"] = (paired_prob_swing["o"] - paired_prob_swing["rusher_o"]).abs()
paired_prob_swing["dx"] = paired_prob_swing["x"] - paired_prob_swing["rusher_x"]
paired_prob_swing["dy"] = paired_prob_swing["y"] - paired_prob_swing["rusher_y"]
paired_prob_swing["speed_diff"] = paired_prob_swing["s"] - paired_prob_swing["rusher_s"]

In [ ]:
class BlockOutcomeLSTM(nn.Module):
    def __init__(self, n_features, hidden_size = 32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first = True)
        self.fc1 = nn.Linear(hidden_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted = False
        )
        _, (hidden, _) = self.lstm(packed)
        out = self.relu(self.fc1(hidden[-1]))
        out = self.fc2(out)
        return out.squeeze(1)

device = torch.device("mps")
seq_feature_cols = ["dx", "dy", "distance", "closing_speed", "orientation_diff", "speed_diff", "s", "a"]

model = BlockOutcomeLSTM(n_features=len(seq_feature_cols)).to(device)
model.load_state_dict(torch.load("../models/lstm_sequence.pt", map_location=device))
model.eval()

def get_live_win_probabilities(sequence_df, feature_cols, model, calibrator, device, max_len = 54):
    sequence_df = sequence_df.sort_values("frameId")
    full_seq = sequence_df[feature_cols].values.astype("float32")
    n_frames = len(full_seq)

    probs = []
    with torch.no_grad():
        for t in range(1, n_frames + 1):
            partial = np.zeros((max_len, len(feature_cols)), dtype = "float32")
            length = min(t, max_len)
            partial[:length] = full_seq[:length]

            X_input = torch.tensor(partial[np.newaxis, ...], dtype = torch.float32).to(device)
            len_input = torch.tensor([length], dtype=torch.int64)

            logit = model(X_input, len_input)
            raw_prob = torch.sigmoid(logit).cpu().item()
            calibrated_prob = calibrator.predict_proba([[raw_prob]])[0, 1]
            probs.append(1 - calibrated_prob)

    return probs

swing_results = []
for key, group in paired_prob_swing.groupby(["gameId", "playId", "nflId", "rusher_nflId"]):
    win_probs = get_live_win_probabilities(group, seq_feature_cols, model, calibrator, device)
    prob_range = max(win_probs) - min(win_probs)
    swing_results.append({
        "gameId": key[0], "playId": key[1], "nflId": key[2], "rusher_nflId": key[3],
        "prob_range": prob_range,
    })

swing_results_df = pd.DataFrame(swing_results).sort_values("prob_range", ascending = False)

print(swing_results_df["prob_range"].describe())

In [ ]:
top_swings_full = swing_results_df.merge(
    selection_pool[["gameId","playId","nflId","rusher_nflId","blocker_position"]],
    on=["gameId","playId","nflId","rusher_nflId"], how="left",
)

prob_swings_selected = pick_spread_by_position(top_swings_full, 10)
print(prob_swings_selected[["nflId", "rusher_nflId", "prob_range"]])

In [ ]:
final_selection = pd.concat([
    base_selection,
    prob_swings_selected.merge(
        selection_pool, on=["gameId", "playId", "nflId", "rusher_nflId", "blocker_position"], how = "left"
    ).assign(category = "prob_swing")
]).drop_duplicates(subset=["gameId", "playId", "nflId", "rusher_nflId"]).reset_index(drop = True)

print(final_selection["category"].value_counts())
print(f"\nPosition spread: \n{final_selection['blocker_position'].value_counts()}")

In [ ]:
final_keys = set(zip(
    final_selection["gameId"], final_selection["playId"],
    final_selection["nflId"], final_selection["rusher_nflId"]
))

final_play_keys = set(zip(final_selection["gameId"], final_selection["playId"]))

full_formation = tracking_df[
    tracking_df.apply(lambda r: (r["gameId"], r["playId"]) in final_play_keys, axis = 1)
    ].copy()

print(len(full_formation))
print(f"Distinct plays covered: {full_formation.groupby(['gameId','playId']).ngroups}")

In [ ]:
players = pd.read_csv("../nfl-big-data-bowl-2023/players.csv")

full_formation = full_formation.merge(
    players[["nflId", "displayName", "officialPosition"]],
    on="nflId", how="left",
)

print(full_formation["pff_role"].value_counts(dropna=False))
print(full_formation["playDirection"].value_counts(dropna=False))

In [ ]:
final_paired = full_formation[full_formation["pff_role"].isin(["Pass Block", "Pass Rush"])].copy()

play_animations = []

for _, row in final_selection.iterrows():
    game_id, play_id, nfl_id, rusher_nfl_id = row["gameId"], row["playId"], row["nflId"], row["rusher_nflId"]

    blocker_frames = full_formation[
        (full_formation["gameId"] == game_id) & (full_formation["playId"] == play_id) &
        (full_formation["nflId"] == nfl_id)
    ].sort_values("frameId")

    rusher_frames = full_formation[
        (full_formation["gameId"] == game_id) & (full_formation["playId"] == play_id) &
        (full_formation["nflId"] == rusher_nfl_id)
    ].sort_values("frameId")

    merged = blocker_frames.merge(
        rusher_frames[["frameId", "x", "y", "s", "a", "o", "dir"]].rename(columns={
            "x": "rusher_x", "y": "rusher_y", "s": "rusher_s", "a": "rusher_a",
            "o": "rusher_o", "dir": "rusher_dir",
        }),
        on="frameId", how="inner",
    )

    merged["distance"] = np.sqrt((merged["x"] - merged["rusher_x"])**2 + (merged["y"] - merged["rusher_y"])**2)
    merged["prev_distance"] = merged["distance"].shift(1)
    merged["closing_speed"] = (merged["prev_distance"] - merged["distance"]).fillna(0)
    merged["orientation_diff"] = (merged["o"] - merged["rusher_o"]).abs()
    merged["dx"] = merged["x"] - merged["rusher_x"]
    merged["dy"] = merged["y"] - merged["rusher_y"]
    merged["speed_diff"] = merged["s"] - merged["rusher_s"]

    win_probs = get_live_win_probabilities(merged, seq_feature_cols, model, calibrator, device)

    play_frames = full_formation[
        (full_formation["gameId"] == game_id) & (full_formation["playId"] == play_id)
    ].sort_values("frameId")

    frames_by_id = []
    for i, frame_id in enumerate(merged["frameId"]):
        players_this_frame = play_frames[play_frames["frameId"] == frame_id][
            ["nflId", "displayName", "officialPosition", "pff_role", "x", "y"]
        ].fillna({"displayName": "", "officialPosition": ""}).to_dict(orient="records")

        frames_by_id.append({
            "frameId": int(frame_id),
            "players": players_this_frame,
            "winProbability": round(win_probs[i], 4),
        })

    direction = play_frames["playDirection"].iloc[0]

    starting_frame = {
        "frameId": -1,
        "players": frames_by_id[0]["players"],
        "winProbability": 0.5,
    }

    play_animations.append({
        "gameId": int(game_id), "playId": int(play_id),
        "nflId": int(nfl_id), "rusherNflId": int(rusher_nfl_id),
        "playDirection": direction,
        "blockerName": row["blockerName"], "rusherName": row["rusherName"],
        "blockerTeam": row["team"], "rusherTeam": row["defensiveTeam"],
        "week": int(row["week"]), "category": row["category"],
        "actualOutcome": "loss" if row["blocker_lost_actual"] == 1 else "win",
        "frames": [starting_frame] + frames_by_id,
    })

print(f"Built full-formation animation data for {len(play_animations)} plays")
print(f"Example - frames in first play: {len(play_animations[0]['frames'])}, players in one frame: {len(play_animations[0]['frames'][1]['players'])}")

In [ ]:
with open ("../dashboard/data/play_animator_data.json", "w") as f:
    json.dump(play_animations, f, indent=2)

print(f"Saved {len(play_animations)} animated plays to ../dashboard/data/play_animator_data.json")